# Análisis exploratorio del conjunto BC_cardiotox

Este *notebook* contiene el análisis exploratorio y descriptivo del conjunto de
datos de cardiotoxicidad en pacientes de cáncer de mama. Se corresponde con el
capítulo *Análisis exploratorio* de la memoria.

**Índice**
1. Carga y limpieza de los datos
2. Valores faltantes
3. Análisis univariante (objetivo, variables continuas y binarias)
4. Análisis bivariante respecto a la cardiotoxicidad
5. Análisis multivariante (correlación y PCA)
6. Análisis exploratorio de supervivencia (Kaplan-Meier)
7. La señal electrocardiográfica como dato funcional

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
PINK, BLUE = "#C5006E", "#4C9BD4"
PAL = {0: BLUE, 1: PINK}   # 0 = no CTRCD, 1 = CTRCD

## 1. Carga y limpieza de los datos

Los ficheros utilizan `;` como separador de campos y `,` como separador decimal
(formato europeo). El conjunto *clínico* contiene una fila por paciente; el
conjunto *funcional* contiene, para el subconjunto de pacientes con imagen, la
curva del ECG discretizada en 1001 puntos (`t 1`, `t 2`, ...).

In [ ]:
df = pd.read_csv("BC_cardiotox_clinical_variables.csv", sep=";", decimal=",")
fn = pd.read_csv("BC_cardiotox_functional_variable.csv", sep=";", decimal=",")
print("Clínico:", df.shape, "| Funcional:", fn.shape)
df.head()

In [ ]:
# Grupos de variables
CONT = ["age", "weight", "height", "heart_rate", "LVEF", "PWT",
        "LAd", "LVDd", "LVSd"]
BIN = ["heart_rhythm", "AC", "antiHER2", "HTA", "DL", "DM", "smoker",
       "exsmoker", "ACprev", "antiHER2prev", "RTprev", "CIprev",
       "ICMprev", "ARRprev", "VALVprev", "cxvalv"]
TARGET, TIME = "CTRCD", "time"

LAB = {"age": "Edad (años)", "weight": "Peso (kg)", "height": "Estatura (cm)",
       "heart_rate": "Ritmo cardíaco (lpm)", "LVEF": "FEVI (%)",
       "PWT": "PWT (cm)", "LAd": "LAd (cm)", "LVDd": "LVDd (cm)",
       "LVSd": "LVSd (cm)", "time": "Tiempo de seguimiento (días)",
       "heart_rhythm": "Fibrilación auricular", "AC": "Antraciclinas",
       "antiHER2": "Terapia anti-HER2", "HTA": "Hipertensión",
       "DL": "Dislipidemia", "DM": "Diabetes mellitus", "smoker": "Fumadora",
       "exsmoker": "Exfumadora", "ACprev": "Antraciclinas previas",
       "antiHER2prev": "Anti-HER2 previa", "RTprev": "Radioterapia previa",
       "CIprev": "Insuf. cardíaca previa",
       "ICMprev": "Miocardiopatía isq. previa", "ARRprev": "Arritmia previa",
       "VALVprev": "Valvulopatía previa", "cxvalv": "Cirugía valvular"}

N = len(df)
print(f"N = {N} pacientes | eventos CTRCD = {int(df[TARGET].sum())} "
      f"({100*df[TARGET].mean():.1f}%)")
df.info()

## 2. Valores faltantes

Estudiamos la cantidad y el patrón de valores ausentes. La mayor parte se
concentra en las variables binarias de tratamiento y antecedentes.

In [ ]:
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_pct = (miss / N * 100).round(1)
display(pd.concat([miss, miss_pct], axis=1, keys=["n", "%"]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([LAB.get(v, v) for v in miss_pct.index], miss_pct.values, color=PINK)
ax.invert_yaxis(); ax.set_xlabel("% de valores faltantes")
ax.set_title("Valores faltantes por variable"); plt.show()

## 3. Análisis univariante

### 3.1. Variable objetivo y tiempo de seguimiento

La variable objetivo `CTRCD` está fuertemente desbalanceada. El `time` registra
el tiempo hasta el evento (si `CTRCD=1`) o hasta la censura (si `CTRCD=0`).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
vc = df[TARGET].value_counts().sort_index()
ax[0].bar(["No CTRCD", "CTRCD"], vc.values, color=[BLUE, PINK])
ax[0].set_title("Distribución de la cardiotoxicidad"); ax[0].set_ylabel("Pacientes")
for k in [0, 1]:
    ax[1].hist(df.loc[df[TARGET]==k, TIME], bins=30, alpha=.65, color=PAL[k],
               label=("Evento" if k else "Censurado"))
ax[1].set_title("Tiempo hasta evento / censura"); ax[1].set_xlabel("Días")
ax[1].legend(); plt.show()
print("Mediana de seguimiento:", df[TIME].median(), "días")

### 3.2. Variables continuas

Estadísticos descriptivos, histogramas y diagramas de caja. Se identifican
valores atípicos mediante la regla del rango intercuartílico (1.5·IQR).

In [ ]:
df[CONT + [TIME]].describe().T

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, v in zip(axes.ravel(), CONT):
    s = df[v].dropna()
    ax.hist(s, bins=25, color=BLUE, edgecolor="white", alpha=.85)
    ax.axvline(s.mean(), color=PINK, ls="--"); ax.set_title(LAB[v])
fig.suptitle("Histogramas de las variables continuas"); fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, v in zip(axes.ravel(), CONT):
    ax.boxplot(df[v].dropna(), patch_artist=True,
               boxprops=dict(facecolor=BLUE, alpha=.7),
               medianprops=dict(color=PINK, lw=2))
    ax.set_title(LAB[v]); ax.set_xticks([])
fig.suptitle("Diagramas de caja"); fig.tight_layout(); plt.show()

print("Outliers (1.5*IQR):")
for v in CONT:
    s = df[v].dropna(); q1, q3 = s.quantile(.25), s.quantile(.75); iqr = q3-q1
    n = ((s < q1-1.5*iqr) | (s > q3+1.5*iqr)).sum()
    print(f"  {v:12s}: {n}")

### 3.3. Variables binarias

Prevalencia de los factores de riesgo y tratamientos.

In [ ]:
freq = pd.DataFrame({
    "n": [df[v].notna().sum() for v in BIN],
    "n(=1)": [int(df[v].dropna().sum()) for v in BIN],
    "prev_%": [round(100*df[v].dropna().mean(), 1) for v in BIN]},
    index=[LAB[v] for v in BIN])
display(freq)

prev = pd.Series({LAB[v]: 100*df[v].dropna().mean() for v in BIN}).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(prev.index, prev.values, color=PINK)
ax.set_xlabel("Prevalencia (%)"); ax.set_title("Prevalencia de variables binarias")
plt.show()

## 4. Análisis bivariante respecto a la cardiotoxicidad

### 4.1. Variables continuas (prueba U de Mann-Whitney)

Comparamos la distribución de cada variable continua entre los grupos con y sin
cardiotoxicidad. Se usa una prueba no paramétrica por la asimetría de algunas
variables.

In [ ]:
rows = []
for v in CONT:
    g0 = df.loc[df[TARGET]==0, v].dropna(); g1 = df.loc[df[TARGET]==1, v].dropna()
    _, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    rows.append([LAB[v], f"{g0.mean():.2f}±{g0.std():.2f}",
                 f"{g1.mean():.2f}±{g1.std():.2f}", round(p, 4),
                 "*" if p < .05 else ""])
display(pd.DataFrame(rows, columns=["Variable", "No CTRCD", "CTRCD", "p", "sig"]))

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, v in zip(axes.ravel(), CONT):
    data = [df.loc[df[TARGET]==0, v].dropna(), df.loc[df[TARGET]==1, v].dropna()]
    bp = ax.boxplot(data, patch_artist=True, labels=["No", "Sí"])
    for patch, k in zip(bp["boxes"], [0, 1]):
        patch.set_facecolor(PAL[k]); patch.set_alpha(.7)
    ax.set_title(LAB[v]); ax.set_xlabel("CTRCD")
fig.suptitle("Variables continuas según cardiotoxicidad"); fig.tight_layout(); plt.show()

### 4.2. Variables binarias (tasa de evento, OR y test de independencia)

In [ ]:
rows = []
for v in BIN:
    sub = df[[v, TARGET]].dropna()
    a = ((sub[v]==1)&(sub[TARGET]==1)).sum(); b = ((sub[v]==1)&(sub[TARGET]==0)).sum()
    c = ((sub[v]==0)&(sub[TARGET]==1)).sum(); d = ((sub[v]==0)&(sub[TARGET]==0)).sum()
    t = np.array([[a, b], [c, d]])
    r1 = 100*a/(a+b) if a+b else np.nan; r0 = 100*c/(c+d) if c+d else np.nan
    aa, bb, cc, dd = (t+0.5).ravel() if (t==0).any() else t.ravel()
    OR = (aa*dd)/(bb*cc)
    _, pchi, _, exp = stats.chi2_contingency(t)
    p = stats.fisher_exact(t)[1] if (exp < 5).any() else pchi
    rows.append([LAB[v], round(r0, 1), round(r1, 1), round(OR, 2),
                 round(p, 4), "*" if p < .05 else ""])
display(pd.DataFrame(rows, columns=["Variable", "tasa(=0)%", "tasa(=1)%",
                                    "OR", "p", "sig"]))

## 5. Análisis multivariante

### 5.1. Matriz de correlación (Spearman)

In [ ]:
corr = df[CONT].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True,
            xticklabels=[LAB[v] for v in CONT],
            yticklabels=[LAB[v] for v in CONT], ax=ax, annot_kws={"size": 8})
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("Correlación de Spearman"); plt.show()

### 5.2. Análisis de componentes principales (PCA)

In [ ]:
Xc = df[CONT].dropna(); yc = df.loc[Xc.index, TARGET]
Z = StandardScaler().fit_transform(Xc)
pca = PCA(n_components=2).fit(Z); P = pca.transform(Z)
fig, ax = plt.subplots(figsize=(7, 5.5))
for k in [0, 1]:
    m = (yc==k).values
    ax.scatter(P[m, 0], P[m, 1], s=22, alpha=.6, color=PAL[k],
               label=("CTRCD" if k else "No CTRCD"))
ax.set_xlabel(f"CP1 ({100*pca.explained_variance_ratio_[0]:.1f}%)")
ax.set_ylabel(f"CP2 ({100*pca.explained_variance_ratio_[1]:.1f}%)")
ax.legend(); ax.set_title("PCA de las variables continuas"); plt.show()
print("Varianza explicada CP1+CP2:",
      f"{100*pca.explained_variance_ratio_[:2].sum():.1f}%")

## 6. Análisis exploratorio de supervivencia (Kaplan-Meier)

El estimador de Kaplan-Meier describe la probabilidad de no haber sufrido el
evento a lo largo del tiempo, teniendo en cuenta la censura. El aplanamiento de
la curva sugiere la existencia de una fracción de pacientes "curadas".

In [ ]:
kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(7.5, 5))
kmf.fit(df[TIME], df[TARGET], label="Cohorte global")
kmf.plot_survival_function(ax=ax, color=PINK)
ax.set_ylim(.7, 1.005); ax.set_xlabel("Tiempo (días)")
ax.set_ylabel("Prob. de no sufrir cardiotoxicidad")
ax.set_title("Kaplan-Meier global"); plt.show()

In [ ]:
sel = ["AC", "antiHER2", "HTA", "heart_rhythm"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, v in zip(axes.ravel(), sel):
    sub = df[[TIME, TARGET, v]].dropna()
    for k in [0, 1]:
        m = sub[v]==k
        if m.sum():
            kmf.fit(sub.loc[m, TIME], sub.loc[m, TARGET], label=f"{LAB[v]}={int(k)}")
            kmf.plot_survival_function(ax=ax, color=PAL[k], ci_show=False)
    lr = logrank_test(sub.loc[sub[v]==0, TIME], sub.loc[sub[v]==1, TIME],
                      sub.loc[sub[v]==0, TARGET], sub.loc[sub[v]==1, TARGET])
    ax.set_title(f"{LAB[v]} (log-rank p={lr.p_value:.3f})"); ax.set_ylim(.55, 1.005)
fig.suptitle("Kaplan-Meier por subgrupos"); fig.tight_layout(); plt.show()

## 7. La señal electrocardiográfica como dato funcional

La curva del ECG se extrae de cada imagen médica (traza verde) y se discretiza en
1001 puntos. Aquí cargamos las señales ya extraídas (`senales_ecg_full.csv`,
generadas con `extraer_ecg.py`) y las visualizamos por grupo.

In [ ]:
import re
ecg = pd.read_csv("RESULTADOS_ECG/senales_ecg_full.csv")
pcols = [c for c in ecg.columns if re.match(r"p\d+$", c)]
E = ecg[pcols].to_numpy(float); yE = ecg[TARGET].to_numpy()
xg = np.linspace(0, 1, E.shape[1])
print(f"{len(E)} curvas ECG | No CTRCD={int((yE==0).sum())}, CTRCD={int((yE==1).sum())}")

rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, k, title in zip(axes, [0, 1], ["No CTRCD", "CTRCD"]):
    idx = np.where(yE==k)[0]
    for i in rng.choice(idx, size=min(25, len(idx)), replace=False):
        ax.plot(xg, E[i], color=PAL[k], alpha=.25, lw=.8)
    ax.plot(xg, E[idx].mean(0), color="black", lw=2, label="Media")
    ax.set_title(f"{title} (n={len(idx)})"); ax.set_xlabel("Tiempo normalizado")
    ax.legend()
axes[0].set_ylabel("Amplitud ECG"); fig.suptitle("Muestra de curvas ECG por grupo")
fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for k in [0, 1]:
    idx = yE==k; mu = E[idx].mean(0); sd = E[idx].std(0)
    ax.plot(xg, mu, color=PAL[k], lw=2, label=("CTRCD" if k else "No CTRCD"))
    ax.fill_between(xg, mu-sd, mu+sd, color=PAL[k], alpha=.18)
ax.set_xlabel("Tiempo normalizado"); ax.set_ylabel("Amplitud ECG")
ax.set_title("Curva ECG media por grupo (±1 DT)"); ax.legend(); plt.show()

### 7.1. Representación funcional mediante FPCA

Reducimos la dimensión de la curva (1001 puntos) mediante un análisis de
componentes principales funcional, reproduciendo en Python el enfoque del código
en R (`fda::pca.fd`). Cada paciente queda representado por unos pocos *scores*.

In [ ]:
NHARM = 8
fpca = PCA(n_components=NHARM).fit(E)
vr = fpca.explained_variance_ratio_; harm = fpca.components_
scoresF = fpca.transform(E); mean_curve = E.mean(0)
print("Var. explicada (%):", np.round(100*vr, 1))
print("Var. acumulada (%):", np.round(100*np.cumsum(vr), 1))

# Varianza explicada
fig, ax = plt.subplots(figsize=(7, 4.5)); c = np.arange(1, NHARM+1)
ax.bar(c, 100*vr, color=BLUE, alpha=.8, label="Individual")
ax.plot(c, 100*np.cumsum(vr), "o-", color=PINK, label="Acumulada")
ax.axhline(90, color="gray", ls="--"); ax.set_xlabel("Componente")
ax.set_ylabel("Varianza explicada (%)"); ax.legend()
ax.set_title("Varianza explicada por la FPCA del ECG"); plt.show()

In [ ]:
# Modos de variación de las 3 primeras componentes
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for j, ax in enumerate(axes):
    d = 2*np.sqrt(fpca.explained_variance_[j])
    ax.plot(xg, mean_curve, "k", lw=2, label="Media")
    ax.plot(xg, mean_curve+d*harm[j], "--", color=PINK, label="+ PC")
    ax.plot(xg, mean_curve-d*harm[j], ":", color=BLUE, label="- PC")
    ax.set_title(f"Componente {j+1} ({100*vr[j]:.1f}%)"); ax.set_xlabel("Tiempo norm.")
axes[0].legend(fontsize=8); axes[0].set_ylabel("Amplitud ECG")
fig.suptitle("FPCA del ECG: modos de variación"); fig.tight_layout(); plt.show()

In [ ]:
# Scores PC1 vs PC2 por grupo
fig, ax = plt.subplots(figsize=(7, 5.5))
for k in [0, 1]:
    m = yE==k
    ax.scatter(scoresF[m, 0], scoresF[m, 1], s=24, alpha=.6, color=PAL[k],
               label=("CTRCD" if k else "No CTRCD"))
ax.set_xlabel(f"FPC1 ({100*vr[0]:.1f}%)"); ax.set_ylabel(f"FPC2 ({100*vr[1]:.1f}%)")
ax.set_title("Scores de las 2 primeras componentes funcionales"); ax.legend(); plt.show()

## Conclusiones del análisis exploratorio

- El problema está **fuertemente desbalanceado** (~10% de eventos) y los datos
  están **censurados**, lo que justifica el uso de análisis de supervivencia.
- Las variables continuas más asociadas a la cardiotoxicidad son la **edad**, el
  **ritmo cardíaco**, la **FEVI** (menor en quienes sufren el evento) y el
  **LVSd**. Entre las binarias destaca el **tratamiento previo con antraciclinas**.
- El **aplanamiento de la curva de Kaplan-Meier** evidencia una fracción de
  pacientes no susceptibles, lo que respalda los **modelos de curación**.
- La **señal ECG** se resume eficazmente mediante **FPCA** (la 1ª componente
  explica ~90% de la varianza), lo que la hace manejable como variable funcional
  en la modelización mediante una metodología *single-index*.